# Lynceus Fraud Detection System

# Feature Engineering & Preprocessing Pipeline

### Objective

This notebook creates the preprocessing pipeline that will be used for every model.

The pipeline must guarantee that the exact same preprocessing is applied during:

- Model Training
- Validation
- Testing
- Production Inference (FastAPI)

The preprocessing pipeline will be saved using Joblib.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

import joblib

# Dataset Paths

In [2]:
DATA_DIR = Path("/Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/data/synthetic")

TRAIN_PATH = DATA_DIR / "transactions_train.csv"
VAL_PATH = DATA_DIR / "transactions_val.csv"
TEST_PATH = DATA_DIR / "transactions_test.csv"

PROCESSED_DIR = Path("/Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/data/processed")

ARTIFACT_DIR = Path("/Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/artifacts")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Load Datasets

In [3]:
train_df = pd.read_csv(
    TRAIN_PATH,
    parse_dates=["timestamp"]
)

val_df = pd.read_csv(
    VAL_PATH,
    parse_dates=["timestamp"]
)

test_df = pd.read_csv(
    TEST_PATH,
    parse_dates=["timestamp"]
)

In [4]:
print("Train :", train_df.shape)
print("Validation :", val_df.shape)
print("Test :", test_df.shape)

Train : (70000, 23)
Validation : (15000, 23)
Test : (15000, 23)


# Dataset Validation

In [5]:
assert list(train_df.columns) == list(val_df.columns)

assert list(train_df.columns) == list(test_df.columns)

print("All datasets have identical schema.")

All datasets have identical schema.


In [6]:
display(train_df.head())

,timestamp,amount,currency,payment_method,hour,day_of_week,is_weekend,sender_account_age_days,receiver_account_age_days,sender_txn_count_24h,...,is_new_receiver,origin_country,destination_country,merchant_category,device_type,device_trusted,cross_border,high_risk_country,velocity_score,fraud_label
0,2025-01-01 00:02:41,1576.01,GBP,WALLET,0,2,False,371,1779,8,...,False,GB,GB,SHOPPING,ANDROID,False,False,False,0.119,0
1,2025-01-01 00:07:42,1583.34,INR,WALLET,0,2,False,1524,4916,6,...,False,IN,IN,UTILITIES,ANDROID,True,False,False,0.081,0
2,2025-01-01 00:28:15,1389.31,INR,WALLET,0,2,False,1382,1751,4,...,False,IN,IN,SHOPPING,IOS,True,False,False,0.037,0
3,2025-01-01 00:28:53,1859.82,INR,WALLET,0,2,False,1585,2010,8,...,False,IN,IN,ENTERTAINMENT,ANDROID,True,False,False,0.184,0
4,2025-01-01 00:30:30,1218.01,USD,UPI,0,2,False,179,921,13,...,False,AE,AE,FOOD,ANDROID,True,False,False,0.262,0


# Separate Features & Target

In [7]:
TARGET = "fraud_label"

X_train = train_df.drop(columns=TARGET)
y_train = train_df[TARGET]

X_val = val_df.drop(columns=TARGET)
y_val = val_df[TARGET]

X_test = test_df.drop(columns=TARGET)
y_test = test_df[TARGET]

In [8]:
print(X_train.shape)
print(y_train.shape)

(70000, 22)
(70000,)


# Drop Timestamp

The generator has already extracted:

- hour
- day_of_week
- is_weekend

Therefore the original timestamp is unnecessary for the first model.

In [9]:
X_train = X_train.drop(columns="timestamp")

X_val = X_val.drop(columns="timestamp")

X_test = X_test.drop(columns="timestamp")

# Feature Categories

In [11]:
numeric_features = [

    "amount",

    "hour",

    "day_of_week",

    "sender_account_age_days",

    "receiver_account_age_days",

    "sender_txn_count_24h",

    "receiver_txn_count_24h",

    "sender_avg_amount_30d",

    "receiver_avg_amount_30d",

    "velocity_score"

]

In [12]:
binary_features = [

    "is_weekend",

    "is_new_receiver",

    "device_trusted",

    "cross_border",

    "high_risk_country"

]

In [13]:
low_cardinality_features = [

    "currency",

    "payment_method",

    "merchant_category",

    "device_type"

]

In [14]:
high_cardinality_features = [

    "origin_country",

    "destination_country"

]

In [15]:
for df in [X_train, X_val, X_test]:
    df[binary_features] = df[binary_features].astype("int8")

In [16]:
all_features = (

    numeric_features

    + binary_features

    + low_cardinality_features

    + high_cardinality_features

)

In [17]:
missing = set(X_train.columns) - set(all_features)

extra = set(all_features) - set(X_train.columns)

print("Missing Features :", missing)

print("Extra Features :", extra)

Missing Features : set()
Extra Features : set()


# Numerical Pipeline

In [18]:
numeric_pipeline = Pipeline(

    steps=[

        (
            "imputer",

            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",

            StandardScaler()
        )

    ]

)

# Binary Pipeline

In [19]:
binary_pipeline = Pipeline(

    steps=[

        (
            "imputer",

            SimpleImputer(
                strategy="most_frequent"
            )
        )

    ]

)

# Categorical Pipeline (Low Cardinality)

Low-cardinality categorical features are encoded using
One-Hot Encoding.

In [20]:
categorical_pipeline = Pipeline(

    steps=[

        (
            "imputer",

            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",

            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )

    ]

)

# High Cardinality Pipeline

Country features are encoded using Ordinal Encoding.

Unknown categories during inference will be encoded as -1.

In [21]:
high_cardinality_pipeline = Pipeline(

    steps=[

        (
            "imputer",

            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",

            OrdinalEncoder(

                handle_unknown="use_encoded_value",

                unknown_value=-1

            )
        )

    ]

)

# Column Transformer

In [22]:
preprocessor = ColumnTransformer(

    transformers=[

        (

            "numeric",

            numeric_pipeline,

            numeric_features

        ),

        (

            "binary",

            binary_pipeline,

            binary_features

        ),

        (

            "categorical",

            categorical_pipeline,

            low_cardinality_features

        ),

        (

            "country",

            high_cardinality_pipeline,

            high_cardinality_features

        )

    ],

    remainder="drop"

)

# Fit Preprocessor

In [23]:
preprocessor.fit(X_train)

print("Pipeline fitted successfully.")

Pipeline fitted successfully.


# Transform Datasets

In [24]:
X_train_processed = preprocessor.transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

In [25]:
print(X_train_processed.shape)
print(X_val_processed.shape)
print(X_test_processed.shape)

(70000, 41)
(15000, 41)
(15000, 41)


# Feature Names

In [26]:
feature_names = preprocessor.get_feature_names_out()

len(feature_names)

41

In [27]:
feature_names

array(['numeric__amount', 'numeric__hour', 'numeric__day_of_week',
       'numeric__sender_account_age_days',
       'numeric__receiver_account_age_days',
       'numeric__sender_txn_count_24h', 'numeric__receiver_txn_count_24h',
       'numeric__sender_avg_amount_30d',
       'numeric__receiver_avg_amount_30d', 'numeric__velocity_score',
       'binary__is_weekend', 'binary__is_new_receiver',
       'binary__device_trusted', 'binary__cross_border',
       'binary__high_risk_country', 'categorical__currency_EUR',
       'categorical__currency_GBP', 'categorical__currency_INR',
       'categorical__currency_USD',
       'categorical__payment_method_BANK_TRANSFER',
       'categorical__payment_method_CARD',
       'categorical__payment_method_UPI',
       'categorical__payment_method_WALLET',
       'categorical__merchant_category_EDUCATION',
       'categorical__merchant_category_ELECTRONICS',
       'categorical__merchant_category_ENTERTAINMENT',
       'categorical__merchant_category_

# Convert to DataFrame

In [28]:
X_train_processed = pd.DataFrame(

    X_train_processed,

    columns=feature_names,

    index=X_train.index

)

X_val_processed = pd.DataFrame(

    X_val_processed,

    columns=feature_names,

    index=X_val.index

)

X_test_processed = pd.DataFrame(

    X_test_processed,

    columns=feature_names,

    index=X_test.index

)

In [29]:
display(X_train_processed.head())

,numeric__amount,numeric__hour,numeric__day_of_week,numeric__sender_account_age_days,numeric__receiver_account_age_days,numeric__sender_txn_count_24h,numeric__receiver_txn_count_24h,numeric__sender_avg_amount_30d,numeric__receiver_avg_amount_30d,numeric__velocity_score,...,categorical__merchant_category_TRAVEL,categorical__merchant_category_UTILITIES,categorical__device_type_ANDROID,categorical__device_type_IOS,categorical__device_type_LINUX,categorical__device_type_MACOS,categorical__device_type_WEB,categorical__device_type_WINDOWS,country__origin_country,country__destination_country
0,-0.566028,-2.396989,-0.483571,-1.162605,-0.450129,-0.426549,-1.205860,-0.675866,-0.614013,-0.573603,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,3.0
1,-0.565130,-2.396989,-0.483571,-0.226140,1.666232,-0.600652,-1.540238,-0.689212,-0.818113,-0.743943,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0
2,-0.588907,-2.396989,-0.483571,-0.341472,-0.469019,-0.774755,-1.540238,-0.685421,1.825477,-0.941179,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,4.0,4.0
3,-0.531249,-2.396989,-0.483571,-0.176596,-0.294286,-0.426549,-1.289455,-0.642535,0.027128,-0.282233,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0
4,-0.609898,-2.396989,-0.483571,-1.318547,-1.028974,0.008709,-0.035536,-0.685069,-0.341204,0.067412,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
display(X_train_processed.describe())

,numeric__amount,numeric__hour,numeric__day_of_week,numeric__sender_account_age_days,numeric__receiver_account_age_days,numeric__sender_txn_count_24h,numeric__receiver_txn_count_24h,numeric__sender_avg_amount_30d,numeric__receiver_avg_amount_30d,numeric__velocity_score,...,categorical__merchant_category_TRAVEL,categorical__merchant_category_UTILITIES,categorical__device_type_ANDROID,categorical__device_type_IOS,categorical__device_type_LINUX,categorical__device_type_MACOS,categorical__device_type_WEB,categorical__device_type_WINDOWS,country__origin_country,country__destination_country
count,7.000000e+04,7.000000e+04,7.000000e+04,7.000000e+04,7.000000e+04,7.000000e+04,7.000000e+04,7.000000e+04,7.000000e+04,7.000000e+04,...,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,2.030122e-17,-1.291158e-16,9.460369e-17,-6.638499e-17,1.360182e-16,-6.699403e-17,-3.735425e-17,1.161230e-16,1.697182e-16,-4.060244e-19,...,0.111014,0.187071,0.306971,0.177943,0.115286,0.132086,0.129829,0.137886,3.938486,4.532471
std,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,...,0.314152,0.389972,0.461241,0.382467,0.319369,0.338586,0.336117,0.344782,1.288824,2.558104
min,-7.586819e-01,-2.396989e+00,-1.502260e+00,-1.463118e+00,-1.649648e+00,-1.122960e+00,-1.707427e+00,-8.287755e-01,-1.727349e+00,-1.107036e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-5.329023e-01,-5.648615e-01,-9.929155e-01,-7.938667e-01,-8.650357e-01,-6.877030e-01,-8.714818e-01,-6.320932e-01,-8.639237e-01,-6.722211e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,4.000000
50%,-3.600736e-01,1.013666e-01,2.577281e-02,-2.919283e-01,-1.160980e-02,-2.524456e-01,-3.553605e-02,-2.669408e-01,-4.667650e-03,-2.553369e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,4.000000
75%,1.899314e-01,7.675948e-01,1.044461e+00,7.405784e-01,8.499119e-01,2.698633e-01,8.840042e-01,1.083171e-01,8.419960e-01,2.825783e-01,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,4.000000
max,2.415461e+01,1.433823e+00,1.553805e+00,3.344282e+00,2.380007e+00,5.667056e+00,2.305112e+00,5.862357e+00,2.368148e+00,3.375590e+00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,7.000000,12.000000


# Processed Dataset Information

In [31]:
print("Train :", X_train_processed.shape)

print("Validation :", X_val_processed.shape)

print("Test :", X_test_processed.shape)

Train : (70000, 41)
Validation : (15000, 41)
Test : (15000, 41)


In [32]:
print("Missing Values")

print(X_train_processed.isna().sum().sum())

Missing Values
0


In [33]:
assert X_train_processed.isnull().sum().sum() == 0

assert X_val_processed.isnull().sum().sum() == 0

assert X_test_processed.isnull().sum().sum() == 0

print("No missing values after preprocessing.")

No missing values after preprocessing.


# Save Preprocessing Pipeline

The fitted preprocessing pipeline is saved and will be reused during:

- Model Training
- Model Evaluation
- FastAPI Inference

In [34]:
PIPELINE_PATH = ARTIFACT_DIR / "preprocessing_pipeline.joblib"

joblib.dump(
    preprocessor,
    PIPELINE_PATH
)

print(f"Pipeline saved to:\n{PIPELINE_PATH}")

Pipeline saved to:
/Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/artifacts/preprocessing_pipeline.joblib


# Save Processed Datasets

In [35]:
train_processed = X_train_processed.copy()
train_processed[TARGET] = y_train.values

val_processed = X_val_processed.copy()
val_processed[TARGET] = y_val.values

test_processed = X_test_processed.copy()
test_processed[TARGET] = y_test.values

In [36]:
train_processed.to_csv(
    PROCESSED_DIR / "train_processed.csv",
    index=False
)

val_processed.to_csv(
    PROCESSED_DIR / "val_processed.csv",
    index=False
)

test_processed.to_csv(
    PROCESSED_DIR / "test_processed.csv",
    index=False
)

# Verify Saved Files

In [37]:
print("Artifacts")

for file in ARTIFACT_DIR.iterdir():
    print("•", file.name)

print()

print("Processed Data")

for file in PROCESSED_DIR.iterdir():
    print("•", file.name)

Artifacts
• shap
• preprocessing_pipeline.joblib
• reports

Processed Data
• test_processed.csv
• val_processed.csv
• train_processed.csv


# Reload Pipeline

Sanity check to ensure the saved pipeline can be loaded successfully.

In [38]:
loaded_preprocessor = joblib.load(PIPELINE_PATH)

print(type(loaded_preprocessor))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>


# Transform New Sample

In [39]:
sample = X_test.iloc[[0]]

sample_processed = loaded_preprocessor.transform(sample)

print(sample_processed.shape)

(1, 41)


In [40]:
sample_df = pd.DataFrame(
    sample_processed,
    columns=feature_names
)

display(sample_df)

,numeric__amount,numeric__hour,numeric__day_of_week,numeric__sender_account_age_days,numeric__receiver_account_age_days,numeric__sender_txn_count_24h,numeric__receiver_txn_count_24h,numeric__sender_avg_amount_30d,numeric__receiver_avg_amount_30d,numeric__velocity_score,...,categorical__merchant_category_TRAVEL,categorical__merchant_category_UTILITIES,categorical__device_type_ANDROID,categorical__device_type_IOS,categorical__device_type_LINUX,categorical__device_type_MACOS,categorical__device_type_WEB,categorical__device_type_WINDOWS,country__origin_country,country__destination_country
0,0.281659,-0.564862,0.025773,1.303229,1.592696,2.359099,-0.28632,-0.596721,-0.548862,2.5732,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,4.0,4.0


# Notebook Summary

Completed preprocessing pipeline:

- Loaded train/validation/test datasets
- Categorized features
- Built preprocessing pipelines
- Combined pipelines using ColumnTransformer
- Fitted preprocessing on training data
- Transformed all datasets
- Saved preprocessing pipeline
- Saved processed datasets
- Verified pipeline loading

In [41]:
print("=" * 60)
print("Feature Engineering Completed Successfully")
print("=" * 60)

print(f"Training samples   : {len(train_processed):,}")
print(f"Validation samples : {len(val_processed):,}")
print(f"Test samples       : {len(test_processed):,}")

print(f"Features after preprocessing : {X_train_processed.shape[1]}")

print("\nArtifacts saved:")
print(f"- {PIPELINE_PATH}")
print(f"- {PROCESSED_DIR / 'train_processed.csv'}")
print(f"- {PROCESSED_DIR / 'val_processed.csv'}")
print(f"- {PROCESSED_DIR / 'test_processed.csv'}")

Feature Engineering Completed Successfully
Training samples   : 70,000
Validation samples : 15,000
Test samples       : 15,000
Features after preprocessing : 41

Artifacts saved:
- /Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/artifacts/preprocessing_pipeline.joblib
- /Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/data/processed/train_processed.csv
- /Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/data/processed/val_processed.csv
- /Users/lakshaydahiya/Desktop/engineerOS/01_projects/lynceus/backend/app/ml/data/processed/test_processed.csv
